# 02o2: Extracting Data from Multiple NotePlan Files (Bulk Processing)

This notebook demonstrates how to extract entities and relationships from **multiple** NotePlan files in bulk using the graph builder agent. This is useful for processing large numbers of files efficiently with caching and progress tracking.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup

All environment detection, Neo4j connection, and NotePlan directory configuration are handled in `00-import.ipynb`.

## Overview

This notebook focuses on extracting data from **multiple NotePlan files** in bulk. It's perfect for:
- Processing large numbers of files efficiently
- Batch processing with caching support
- Progress tracking across multiple files
- Production workflows that need to process entire directories

**Alternative Approaches:**
- [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb): Extract data from a **single file** for testing/debugging
- [**02o3-extracting-data-langchain.ipynb**](./02o3-extracting-data-langchain.ipynb): Alternative extraction method using LangChain's structured outputs

We'll:
1. Load multiple NotePlan files to process
2. Use graph builder agent to extract entities and relationships from each file
3. Display summary statistics across all files
4. Save extracted data to disk with caching (skips already-processed files)


In [1]:
# Run common imports and setup
# Note: nest_asyncio is automatically enabled in 00-import.ipynb
%run 00-import.ipynb

# Additional imports specific to this notebook
import asyncio
from knowledge_agents.agents.graph_builder_agent import run_graph_builder_agent

# NotePlan utilities
from notes.traversal import get_files_from_last_month
from notes.parser import read_noteplan_file
from notes.filter import should_skip_file

print("✅ Additional libraries imported")


✅ Added to path: /Users/omareid/Workspace/git/knowledge-agents/src
   Project root: /Users/omareid/Workspace/git/knowledge-agents
✅ Standard library imports loaded
✅ Repository components imported
🔍 Runtime detection: local
✅ Settings loaded:
   Neo4j URI: bolt://localhost:7687
   Neo4j Database: knowledge
   Neo4j Username: neo4j
   Neo4j Password: ********
   LiteLLM Proxy Host: localhost

💡 To override settings, see Settings class docstring:
   help(Settings)  # or help(get_settings)
   # Quick examples:
   # settings = get_settings(neo4j_password='your_password')
   # settings = get_settings(runtime_env='container')

📁 NotePlan directory: /Users/omareid/Library/Containers/co.noteplan.NotePlan3/Data/Library/Application Support/co.noteplan.NotePlan3
   Directory exists: True

✅ Successfully connected to Neo4j
✅ Data manipulation libraries imported
✅ Additional libraries imported


## Load Multiple NotePlan Files

Get multiple NotePlan files to process in bulk. Note: `NOTEPLAN_DIR` is already set up in `00-import.ipynb` based on the runtime environment.


In [ ]:
# Get NotePlan files from the last month
files = get_files_from_last_month(NOTEPLAN_DIR)
print(f"Found {len(files)} files to process")

# Filter out files we should skip
files = [(fp, mod_time) for fp, mod_time in files if not should_skip_file(fp)]
print(f"After filtering: {len(files)} valid files")

# Optionally limit the number of files for testing
# Remove or increase this limit for full processing
MAX_FILES = None  # Set to None to process all files, or a number like 10, 50, 100
if MAX_FILES:
    files = files[:MAX_FILES]
    print(f"Limited to first {len(files)} files for processing")

print(f"\n📁 Will process {len(files)} files")
if len(files) > 0:
    print(f"   First file: {files[0][0].relative_to(NOTEPLAN_DIR)}")
    print(f"   Last file: {files[-1][0].relative_to(NOTEPLAN_DIR)}")


Found 176 files to process
After filtering: 176 files
Processing 5 files for this demo


## Extract Entities and Relationships

Use the graph builder agent to extract entities and relationships from each file. Files are processed sequentially with caching support - already-processed files are automatically skipped.


In [ ]:
# Import utilities for data persistence and processing
from knowledge_agents.utils import (
    check_file_cached,
    get_data_dir,
    load_nodes_edges,
    process_file_with_sections_and_embeddings,
    save_nodes_edges,
    save_sections_embeddings,
)
from pathlib import Path
import time

# Set up data directory (build/data/)
project_root = Path(os.getcwd())
if project_root.name == "notebooks":
    project_root = project_root.parent
data_dir = get_data_dir(project_root / "build", NOTEPLAN_DIR)
print(f"📁 Data directory: {data_dir}")

# Process all files with caching and file saving
extracted_data = []
failed_files = []
use_cache = True  # Set to False to regenerate all files
start_time = time.time()

print(f"\n🚀 Starting bulk processing of {len(files)} files...")
print(f"   Caching: {'enabled' if use_cache else 'disabled'}")
print(f"   Progress will be shown below\n")

for idx, (file_path, mod_time) in enumerate(files, 1):
    relative_path = str(file_path.relative_to(NOTEPLAN_DIR))
    
    # Check cache
    is_cached = False
    if use_cache:
        is_cached, nodes_edges_path, _ = check_file_cached(
            data_dir, relative_path, check_nodes_edges=True, check_sections_embeddings=False
        )
        if is_cached:
            print(f"[{idx}/{len(files)}] {relative_path} [cached - skipping]")
            # Load cached data for summary
            if nodes_edges_path and nodes_edges_path.exists():
                cached_data = load_nodes_edges(nodes_edges_path)
                print(f"  ✅ Cached: {len(cached_data.get('entities', []))} entities, {len(cached_data.get('relationships', []))} relationships")
                # Convert to output format for summary
                from knowledge_agents.types.graph import Entity, Relationship, GraphBuilderAgentOutput
                entities = [Entity(**e) for e in cached_data.get('entities', [])]
                relationships = [Relationship(**r) for r in cached_data.get('relationships', [])]
                output = GraphBuilderAgentOutput(entities=entities, relationships=relationships, insights=cached_data.get('insights', []))
                extracted_data.append((relative_path, output))
            continue
    
    print(f"[{idx}/{len(files)}] Processing: {relative_path}")
    
    # Process file: extract nodes/edges and generate sections with embeddings
    # Note: Logging is automatically written to stdout.log and stderr.log files
    try:
        output, sections_with_embeddings = asyncio.run(
            process_file_with_sections_and_embeddings(
                file_path=file_path,
                relative_path=relative_path,
                dependencies=dependencies,
                data_dir=data_dir,
                generate_embeddings_flag=True,
                use_cache=False,  # Already checked above
            )
        )
        
        # Save nodes/edges if extraction succeeded
        if output:
            save_nodes_edges(data_dir, relative_path, output, source_file_path=file_path)
            extracted_data.append((relative_path, output))
            print(f"  ✅ Extracted {len(output.entities)} entities, {len(output.relationships)} relationships")
        else:
            print(f"  ❌ Failed to extract entities/relationships")
            failed_files.append(relative_path)
        
        # Save sections with embeddings
        if sections_with_embeddings:
            save_sections_embeddings(data_dir, relative_path, sections_with_embeddings, source_file_path=file_path)
            print(f"  ✅ Saved {len(sections_with_embeddings)} sections with embeddings")
        
        # Note: stdout.log and stderr.log are automatically created by the logging system
        
    except Exception as e:
        print(f"  ❌ Error processing {relative_path}: {e}")
        failed_files.append(relative_path)
        # Logs are automatically saved even on exception

# Summary
elapsed_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ Processed {len(extracted_data)} files successfully")
print(f"❌ Failed to process {len(failed_files)} files")
print(f"⏱️  Total time: {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
if len(extracted_data) > 0:
    print(f"📊 Average time per file: {elapsed_time/len(extracted_data):.1f} seconds")
print(f"{'='*60}")

if failed_files:
    print(f"\n❌ Failed files:")
    for failed_file in failed_files:
        print(f"   - {failed_file}")

Processing: np-out.log


OPENAI_API_KEY is not set, skipping trace export
Error in graph builder agent for np-out.log: Invalid JSON when parsing { "entities": [ { "name": "2025-03-01 16:09:59 +0000", "type": "Date", "properties": { } }, { "name": "v3.16 (1323)", "type": "AppVersion", "properties": { } }, { "name": "macOS (Version 15.3.1 (Build 24D70))", "type": "OS", "properties": { } }, { "name": "2025-03-01", "type": "Date", "properties": { } }, { "name": "2025-03-01 16:09:52 +0000", "type": "Date", "properties": { } }, { "name": "2025-03-01 16:10:07 +0000", "type": "Date", "properties": { } }, { "name": "DFF38DBF-2293-4D22-A43C-D6DF5CA1A303", "type": "RecordID", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { } }, { "name": "20250301.txt", "type": "FileName", "properties": { 

  ✅ Extracted 0 entities, 0 relationships
Processing: np-error.log


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 12 entities, 11 relationships
Processing: Filters/folders.views


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 8 entities, 5 relationships
Processing: Calendar/20250927.txt


OPENAI_API_KEY is not set, skipping trace export


  ✅ Extracted 4 entities, 5 relationships
Processing: Calendar/20251026.txt
  ✅ Extracted 0 entities, 0 relationships

✅ Processed 5 files successfully


OPENAI_API_KEY is not set, skipping trace export


## Display Extracted Data

View the extracted entities and relationships.


In [ ]:
# Collect and display summary statistics across all files
all_entities = []
all_relationships = []
files_by_entity_count = []
files_by_relationship_count = []

for file_path, output in extracted_data:
    entity_count = len(output.entities)
    relationship_count = len(output.relationships)
    
    files_by_entity_count.append({
        "file": file_path,
        "entities": entity_count,
        "relationships": relationship_count
    })
    
    for entity in output.entities:
        all_entities.append({
            "file": file_path,
            "name": entity.name,
            "type": entity.type,
            "properties": entity.properties
        })
    for rel in output.relationships:
        all_relationships.append({
            "file": file_path,
            "from": rel.from_entity,
            "type": rel.type,
            "to": rel.to_entity,
            "properties": rel.properties
        })

# Display summary statistics
print(f"\n📊 Summary Statistics\n")
print(f"Total files processed: {len(extracted_data)}")
print(f"Total entities extracted: {len(all_entities)}")
print(f"Total relationships extracted: {len(all_relationships)}")

if files_by_entity_count:
    df_files = pd.DataFrame(files_by_entity_count)
    print(f"\n📁 Files by extraction count:")
    print(df_files.sort_values('entities', ascending=False).head(10).to_string(index=False))
    print(f"\n   Average entities per file: {df_files['entities'].mean():.1f}")
    print(f"   Average relationships per file: {df_files['relationships'].mean():.1f}")

# Display entity type distribution
if all_entities:
    entity_types = {}
    for entity in all_entities:
        entity_type = entity['type']
        entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
    
    print(f"\n📈 Entity Type Distribution:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True):
        print(f"   {entity_type}: {count}")

# Display relationship type distribution
if all_relationships:
    rel_types = {}
    for rel in all_relationships:
        rel_type = rel['type']
        rel_types[rel_type] = rel_types.get(rel_type, 0) + 1
    
    print(f"\n🔗 Relationship Type Distribution:")
    for rel_type, count in sorted(rel_types.items(), key=lambda x: x[1], reverse=True):
        print(f"   {rel_type}: {count}")

# Display sample entities
if all_entities:
    print(f"\n✅ Sample Entities (first 20):")
    df_entities = pd.DataFrame(all_entities)
    print(df_entities.head(20).to_string(index=False))

# Display sample relationships
if all_relationships:
    print(f"\n🔗 Sample Relationships (first 20):")
    df_relationships = pd.DataFrame(all_relationships)
    print(df_relationships.head(20).to_string(index=False))


## Next Steps

Now that data is extracted from multiple files, you can:

**Review Results:**
- Check the summary statistics above
- Review individual file logs in `build/data/` directory
- Inspect specific files using [**02o1-extracting-data.ipynb**](./02o1-extracting-data.ipynb) for detailed view

**Continue with Data Storage:**
- [**02-extracting-embeddings.ipynb**](./02-extracting-embeddings.ipynb): Generate vector embeddings from NotePlan notes
- [**03-loading-data.ipynb**](./03-loading-data.ipynb): Load entities and relationships into Neo4j graph
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store vector embeddings in Neo4j
